# LLM4Rec: SASRec to LLaMA Adapter Pipeline
This notebook runs the full pipeline for your ECS172 project on Google Colab.
Make sure you have uploaded your entire project folder (with the CSVs) to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# IMPORTANT: Change this path to where your ECS172 folder is located in your Google Drive
PROJECT_DIR = '/content/drive/MyDrive/ECS172'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
!ls

## Step 0: Install Dependencies
Installs standard requirements and Unsloth (which is highly optimized for Colab GPUs).

In [ ]:
!pip install -r requirements.txt
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## Step 1: Train SASRec
This runs Phase 1 to generate the base collaborative item embeddings and saves them to `checkpoints/`.

In [ ]:
!python scripts/train_sasrec.py --epochs 15

## Step 2: Build Prompts Data
Generates the JSON prompt files needed for evaluation and fine-tuning.

In [ ]:
!python scripts/build_ranking_prompts_json.py --split train --n-candidates 5
!python scripts/build_ranking_prompts_json.py --split test --n-candidates 10

## Step 3: Train the Adapter (LLaMA-Grounded CE)
This runs Phase 2: Training the MLP adapter by teacher-forcing the movie title reconstruction through frozen LLaMA.

*Note: If you want to ground the adapter to your fine-tuned model, change `--llm-model unsloth/Llama-3.2-1B-Instruct` to the path of your fine-tuned model folder (e.g. `./llama31-1b-movielens-full-final`).*

In [ ]:
!python scripts/train_adapter.py --epochs 10 --batch-size 8 --llm-model unsloth/Llama-3.2-1B-Instruct

## Step 4: Evaluate Ranking
Run inference to see how well the grounded adapter performs compared to the text-only baseline.

In [ ]:
print('--- Running Text-Only Baseline ---')
!python scripts/eval_ranking.py --no-injection --model unsloth/Llama-3.2-1B-Instruct

print('\n--- Running with Grounded Adapter Injection ---')
!python scripts/eval_ranking.py --use-adapter-llm --model unsloth/Llama-3.2-1B-Instruct